In [ ]:
print('Hello World!')

Hello World!


In [ ]:
#Step 1 - Get the original link and extract the page numbers
#Step 3 - Go to the search results and extract links to each paper 
#Step 4 - Access the paper and extract the abstract as required

Step 1:

In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re

def get_search_results(response):
    search_results = response.find_all('a',class_='docsum-title')
    page_links = [result.get('href') for result in search_results]
    return page_links

def extract_info(page):
    paper_url='https://pubmed.ncbi.nlm.nih.gov'+page
    response = requests.get(paper_url)
    # Step 4: Check Response Status
    if response.status_code == 200:
        # Step 5: Parse the HTML
        soup = BeautifulSoup(response.content, 'html.parser')

    # Find all <h2> elements with the class 'title'
    #print(paper_url)
    abstract_headings = soup.find_all('div', class_='abstract-content selected')
    if len(abstract_headings) == 0: 
        return paper_url,'No Abstract Available'
    else:
        abstract_content = abstract_headings[0].find_all('p')
        formatted_text = '\n\n'.join([p.get_text(strip=True) for p in abstract_content])
        return paper_url,formatted_text


url = 'https://pubmed.ncbi.nlm.nih.gov/?term=(((Symptom%5BAll%20Fields%5D%20OR%20Sign%5BAll%20Fields%5D%20OR%20Complaint%5BAll%20Fields%5D))%20AND%20(((Chronic%5BAll%20Fields%5D%20OR%20Uncompensated%5BAll%20Fields%5D%20OR%20Persistent%5BAll%20Fields%5D%20OR%20Enduring%5BAll%20Fields%5D%20OR%20Permanent%5BAll%20Fields%5D%20OR%20Recurrent%5BAll%20Fields%5D%20OR%20Continuous%5BAll%20Fields%5D))%20AND%20((Unilateral%5BAll%20Fields%5D)%20AND%20(((Vestibular%5BAll%20Fields%5D)%20AND%20((Hypofunction%5BAll%20Fields%5D%20OR%20Failure%5BAll%20Fields%5D%20OR%20Loss%5BAll%20Fields%5D%20OR%20Deafferentation%5BAll%20Fields%5D%20OR%20Disease%5BAll%20Fields%5D%20OR%20Disorder%5BAll%20Fields%5D%20OR%20Syndrome%5BAll%20Fields%5D%20OR%20Impairment%5BAll%20Fields%5D%20OR%20Dysfunction%5BAll%20Fields%5D)))%20OR%20(Vestibulopathy%5BAll%20Fields%5D)))))&sort=&page=1'
response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')

total_pages = soup.find('label',class_= 'of-total-pages').get_text(strip=True)
pattern = r'\D'
total_pages = int(re.sub(pattern, '', pages))
total_pages
print(total_pages)

df = pd.DataFrame(columns=['URL', 'Abstract'])

#pages = 2 #Just for checking purpose
for page_number in range(1,total_pages+1):
    #print(page_number) 
    new_url = url[:-1] + str(page_number)
    #print(new_url)
    page_response = requests.get(new_url)
    pages = get_search_results(BeautifulSoup(page_response.content, 'html.parser'))
    abstracts = [extract_info(page) for page in pages]
    #print(abstracts)
    for entry in abstracts:
        df = pd.concat([df, pd.DataFrame([entry], columns=['URL', 'Abstract'])], ignore_index=True)
    print("Page",str(page_number),'of',str(total_pages))

df.to_csv("Abstracts_pubMed.csv",index=False,encoding='utf-8')
print("File Saved")




/tmp/ipykernel_7175/3194651424.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


62
Page 1 of ['/34864777/', '/30947180/', '/33867417/', '/31994145/', '/35723133/', '/35138954/', '/31326945/', '/34784642/', '/33249404/', '/26913496/']
Page 2 of ['/27638074/', '/38271551/', '/18675691/', '/33829994/', '/35971266/', '/7874410/', '/15661119/', '/31887751/', '/34294439/', '/9078929/']
Page 3 of ['/19645909/', '/24057827/', '/37483440/', '/30482329/', '/34191310/', '/32909094/', '/31743237/', '/31300426/', '/35796391/', '/36209619/']
Page 4 of ['/34867755/', '/30167723/', '/37661905/', '/35655817/', '/35221509/', '/33309493/', '/20086281/', '/19629427/', '/32400047/', '/32235946/']
Page 5 of ['/28543062/', '/31522490/', '/10719646/', '/31035836/', '/3921902/', '/34921750/', '/36090885/', '/34456848/', '/33044204/', '/38114342/']
Page 6 of ['/32494851/', '/34776883/', '/15080953/', '/20086282/', '/32009345/', '/36056895/', '/35139478/', '/28434022/', '/35595969/', '/36203969/']
Page 7 of ['/31948878/', '/27001256/', '/34889807/', '/33392761/', '/28814637/', '/27638078/',